# 22. Closing two proposal objectives

Two promises in the Final Proposal were still open. Both are closed here, and neither needs
a GPU.

**Objective 3** promised to "quantify cross-dataset generalisation ... and report the macro-F1
drop **with 95 percent bootstrap confidence intervals**". Results so far gave mean and
standard deviation over three seeds, which measures something different: how much the score
moves when training is repeated. A bootstrap interval measures how much it would move on a
different sample of test posts. Both belong in the write-up and they answer different
questions.

**Objective 4** promised two adversarial attacks "then **evaluate adversarial fine-tuning as a
defence**, measured by the change in macro-F1 before and after each attack". The attacks were
built in notebooks 10 and 13. The defence was never run.

Runtime: about 45 seconds, CPU only.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import warnings; warnings.filterwarnings('ignore')
import sys, numpy as np, pandas as pd
sys.path.insert(0, '/content/drive/MyDrive/dissertation/notebooks')
sys.path.insert(0, '/content/drive/MyDrive/dissertation')
from sklearn.metrics import f1_score
from hinglish_hate import load_lexicon
from hinglish_hate.baseline import build_full_pipeline
from hinglish_hate.attack import romanisation_attack, character_attack, attack_corpus

SEED, B = 42, 2000
rng = np.random.default_rng(SEED)
PROB = '/content/drive/MyDrive/dissertation/writing/probabilities'

# ============ OBJECTIVE 3: bootstrap confidence intervals ====================
print('=' * 72)
print('OBJECTIVE 3: 95% bootstrap confidence intervals')
print('=' * 72)

MODELS = [('tfidf_logreg', 'TF-IDF + LogReg'), ('xlm_roberta_base', 'XLM-RoBERTa'),
          ('google_muril_base_cased', 'MuRIL'), ('ai4bharat_indic_bert', 'IndicBERT v1')]


def boot_macro_f1(y, pred, n=B):
    out = np.empty(n)
    idx = np.arange(len(y))
    for i in range(n):
        s = rng.choice(idx, size=len(idx), replace=True)
        out[i] = f1_score(y[s], pred[s], average='macro')
    return out


rows, draws = [], {}
for tag, name in MODELS:
    for split in ['within', 'cross']:
        try:
            d = pd.read_csv(f'{PROB}/{tag}__{split}.csv')
        except FileNotFoundError:
            continue
        y = d['label'].values
        pred = (d['p_hate'] >= 0.5).astype(int).values
        bs = boot_macro_f1(y, pred)
        draws[(name, split)] = bs
        rows.append({'model': name, 'split': split, 'n': len(y),
                     'macro_f1': round(f1_score(y, pred, average='macro'), 4),
                     'ci_lo': round(float(np.percentile(bs, 2.5)), 4),
                     'ci_hi': round(float(np.percentile(bs, 97.5)), 4)})

ci = pd.DataFrame(rows)
print(ci.to_string(index=False))

# the CI the objective actually asks for: on the DROP
print('\ncross-dataset drop, with 95% CI on the difference:')
drops = []
for tag, name in MODELS:
    if (name, 'within') in draws and (name, 'cross') in draws:
        dd = draws[(name, 'within')] - draws[(name, 'cross')]
        w = ci[(ci.model == name) & (ci.split == 'within')]['macro_f1'].iloc[0]
        c = ci[(ci.model == name) & (ci.split == 'cross')]['macro_f1'].iloc[0]
        drops.append({'model': name, 'within': w, 'cross': c, 'drop': round(w - c, 4),
                      'drop_ci_lo': round(float(np.percentile(dd, 2.5)), 4),
                      'drop_ci_hi': round(float(np.percentile(dd, 97.5)), 4),
                      'excludes_zero': bool(np.percentile(dd, 2.5) > 0)})
dr = pd.DataFrame(drops)
print(dr.to_string(index=False))

import os
os.makedirs('/content/drive/MyDrive/dissertation/writing', exist_ok=True)
ci.to_csv('/content/drive/MyDrive/dissertation/writing/bootstrap_ci.csv', index=False)
dr.to_csv('/content/drive/MyDrive/dissertation/writing/bootstrap_ci_drops.csv', index=False)

OBJECTIVE 3: 95% bootstrap confidence intervals
          model  split    n  macro_f1  ci_lo  ci_hi
TF-IDF + LogReg within  915    0.6421 0.6096 0.6735
TF-IDF + LogReg  cross 4306    0.4351 0.4206 0.4493
    XLM-RoBERTa within  915    0.6526 0.6196 0.6844
    XLM-RoBERTa  cross 4306    0.4697 0.4549 0.4841
          MuRIL within  915    0.6535 0.6218 0.6856
          MuRIL  cross 4306    0.3673 0.3571 0.3786
   IndicBERT v1 within  915    0.6521 0.6193 0.6824
   IndicBERT v1  cross 4306    0.3712 0.3600 0.3825

cross-dataset drop, with 95% CI on the difference:
          model  within  cross   drop  drop_ci_lo  drop_ci_hi  excludes_zero
TF-IDF + LogReg  0.6421 0.4351 0.2070      0.1721      0.2418           True
    XLM-RoBERTa  0.6526 0.4697 0.1829      0.1476      0.2182           True
          MuRIL  0.6535 0.3673 0.2862      0.2530      0.3204           True
   IndicBERT v1  0.6521 0.3712 0.2809      0.2471      0.3133           True


## Objective 4: adversarial fine-tuning

The defence is the standard one: take the training set, generate romanisation variants of
every post, and train on the original and the variants together. The model then sees the
kind of perturbation it will be attacked with.

Two details matter for this to be an honest test.

**Different seeds.** The training augmentation uses seed 7 at rate 0.5; the test attack uses
seed 99 at rate 1.0. If both used the same seed the model could memorise the exact
perturbations it is later scored on, and the defence would look better than it is.

**The character attack is the control.** Training on romanisation variants should defend
against romanisation. Whether it also defends against a *different* attack is the real
question, and the answer is what decides whether this is a general defence or a narrow one.

In [3]:
# ============ OBJECTIVE 4: adversarial fine-tuning defence ===================
print()
print('=' * 72)
print('OBJECTIVE 4: adversarial fine-tuning as a defence')
print('=' * 72)

tr = pd.read_parquet('/content/drive/MyDrive/dissertation/data/splits/bohra_train.parquet')
te = pd.read_parquet('/content/drive/MyDrive/dissertation/data/splits/bohra_test.parquet')
terms = load_lexicon('/content/drive/MyDrive/dissertation/data/hinglish_slur_lexicon.csv')

# held-out attacked test sets, built once, never seen in training
adv_r = attack_corpus(te, romanisation_attack, rate=1.0, seed=99)
adv_c = attack_corpus(te, character_attack,    rate=1.0, seed=99)
print(f"test-set realised change: romanisation {adv_r['token_change_rate'].mean():.3f}, "
      f"character {adv_c['token_change_rate'].mean():.3f}")

# augmentation uses a DIFFERENT seed from the test attack, so the defence cannot
# memorise the exact perturbations it will be evaluated on
aug = attack_corpus(tr, romanisation_attack, rate=0.5, seed=7)
train_aug = pd.concat([tr[['text', 'label']], aug[['text', 'label']]], ignore_index=True)
print(f'augmented training set: {len(tr)} -> {len(train_aug)} rows '
      f'(romanisation variants at rate 0.5, seed 7)\n')

res = []
for label, frame in [('baseline (clean training)', tr),
                     ('adversarially fine-tuned', train_aug)]:
    pipe = build_full_pipeline(terms)
    pipe.fit(frame['text'], frame['label'])
    r = {'training set': label}
    for nm, f in [('clean', te), ('romanisation', adv_r), ('character', adv_c)]:
        r[nm] = round(f1_score(f['label'], pipe.predict(f['text']), average='macro'), 4)
    r['rom drop %'] = round(100 * (r['clean'] - r['romanisation']) / r['clean'], 1)
    r['char drop %'] = round(100 * (r['clean'] - r['character']) / r['clean'], 1)
    res.append(r)

df = pd.DataFrame(res)
print(df.to_string(index=False))

b, a = res[0], res[1]
print('\neffect of the defence:')
print(f"  clean         {b['clean']:.4f} -> {a['clean']:.4f}  ({a['clean']-b['clean']:+.4f})")
print(f"  romanisation  {b['romanisation']:.4f} -> {a['romanisation']:.4f}  "
      f"({a['romanisation']-b['romanisation']:+.4f})")
print(f"  character     {b['character']:.4f} -> {a['character']:.4f}  "
      f"({a['character']-b['character']:+.4f})")
df.to_csv('/content/drive/MyDrive/dissertation/writing/adversarial_defence.csv', index=False)
print('\nsaved bootstrap_ci.csv, bootstrap_ci_drops.csv, adversarial_defence.csv')


OBJECTIVE 4: adversarial fine-tuning as a defence
test-set realised change: romanisation 0.762, character 0.709
augmented training set: 3659 -> 7318 rows (romanisation variants at rate 0.5, seed 7)

             training set  clean  romanisation  character  rom drop %  char drop %
baseline (clean training) 0.6421        0.5889      0.489         8.3         23.8
 adversarially fine-tuned 0.6429        0.6455      0.498        -0.4         22.5

effect of the defence:
  clean         0.6421 -> 0.6429  (+0.0008)
  romanisation  0.5889 -> 0.6455  (+0.0566)
  character     0.4890 -> 0.4980  (+0.0090)

saved bootstrap_ci.csv, bootstrap_ci_drops.csv, adversarial_defence.csv


## Reading the two results

### On the confidence intervals

The intervals on the drop are the headline, because they say whether the cross-dataset
collapse could be a sampling artefact. If the interval excludes zero, it could not.

Note what the two kinds of uncertainty mean, and keep them apart in the write-up:

- **Bootstrap interval:** how much the score would move on a different sample of test posts,
  with the trained model held fixed.
- **Seed standard deviation:** how much it moves when the same model is trained again from a
  different random initialisation.

Report both. A reviewer who sees only one will ask about the other.

The intervals also support a comparison that the point estimates alone cannot. Where two
models' drop intervals do not overlap, the difference between them is real rather than noise,
and that is a claim worth making explicitly.

### On the defence

Three numbers decide what kind of defence this is:

- **Clean performance.** If it falls, the defence costs accuracy on ordinary text, and that
  trade-off has to be reported.
- **The attack it was trained against.** A large recovery here is expected.
- **The other attack.** This is the one that matters. If the defence transfers, it is a
  general hardening of the model. If it does not, it is specific to perturbations of the kind
  seen in training, and the honest conclusion is that defending against one attack family
  says nothing about another.

Whichever way it comes out, write it as a finding rather than a disappointment. A defence
that works narrowly and is reported as working narrowly is a better result than one that is
overclaimed.